# Challenge B: TensorFlow (John Buni)

**Group:** 1

## TensorFlow Gradient Descent Implementation

In [1]:
import tensorflow as tf
import numpy as np
import plotly.graph_objects as go

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


## 1. Challenge B — Full-Batch Gradient Descent

The objective is to train a linear regression model using
TensorFlow's GradientTape to learn the relationship y = 2x + 1.

The model uses Mean Squared Error (MSE) as its loss function
and manually updates its weight and bias using gradient descent.

In [2]:
# Reproducible initialization
tf.random.set_seed(0)

# Training dataset: y = 2x + 1
x = tf.constant([[1.0], [2.0], [3.0], [4.0]])
y = tf.constant([[3.0], [5.0], [7.0], [9.0]])

W = tf.Variable(tf.random.normal((1, 1)))
b = tf.Variable(tf.zeros((1,)))

learning_rate = 0.01
epochs = 300
loss_history = []

for epoch in range(epochs):

    # GradientTape records operations involving trainable variables
    with tf.GradientTape() as tape:
        y_hat = tf.matmul(x, W) + b
        loss = tf.reduce_mean(tf.square(y_hat - y))

    # Returns the derivatives of loss with respect to W and b
    dW, db = tape.gradient(loss, [W, b])

    # Update parameters in the negative gradient direction
    W.assign_sub(learning_rate * dW)
    b.assign_sub(learning_rate * db)

    # Record loss after the parameter update
    epoch_loss = tf.reduce_mean(
        tf.square(tf.matmul(x, W) + b - y)
    )

    loss_history.append(float(epoch_loss.numpy()))

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}: Loss = {epoch_loss.numpy():.6f}")

print("\nLearned W:", W.numpy().item())
print("Learned b:", b.numpy().item())
print("True W: 2.0")
print("True b: 1.0")
print("Final loss:", loss_history[-1])

Epoch 50: Loss = 0.069099
Epoch 100: Loss = 0.051198
Epoch 150: Loss = 0.037934
Epoch 200: Loss = 0.028107
Epoch 250: Loss = 0.020826
Epoch 300: Loss = 0.015430

Learned W: 2.1033825874328613
Learned b: 0.6960417628288269
True W: 2.0
True b: 1.0
Final loss: 0.01543037872761488


### 1.1 Training Loss Visualization

In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, epochs + 1)),
    y=loss_history,
    mode="lines",
    name="Full-Batch Loss"
))

fig.update_layout(
    title="TensorFlow Gradient Descent - Training Loss",
    xaxis_title="Epoch",
    yaxis_title="Mean Squared Error",
    template="plotly_white"
)

fig.show()

### 1.2 Learned vs True Parameters

In [4]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name="True Parameters",
    x=["Weight (W)", "Bias (b)"],
    y=[2.0, 1.0]
))

fig.add_trace(go.Bar(
    name="Learned Parameters",
    x=["Weight (W)", "Bias (b)"],
    y=[W.numpy().item(), b.numpy().item()]
))

fig.update_layout(
    title="Learned vs True Model Parameters",
    yaxis_title="Parameter Value",
    barmode="group",
    template="plotly_white"
)

fig.show()

### Observations

- The training loss decreased consistently throughout all 300 epochs,
  reaching a final MSE of 0.015430.
- The learned weight of 2.1034 is close to the true weight of 2.0.
- The learned bias of 0.6960 is approaching the true bias of 1.0.
- The model has not fully converged, but the decreasing loss suggests
  that additional epochs could improve parameter accuracy.
- The learning rate of 0.01 produces stable updates without divergence.

## 2. TensorFlow Extension Challenge

This experiment investigates how random noise and mini-batch
training affect gradient descent convergence.

Three configurations are compared:

- Full-batch training with clean data.
- Full-batch training with noisy data.
- Mini-batch training with noisy data.

In [5]:
# Generate reproducible random noise
noise = tf.random.stateless_normal(
    shape=y.shape,
    seed=[42, 0],
    mean=0.0,
    stddev=0.5
)

y_noisy = y + noise

print("Original targets:", y.numpy().flatten())
print("Random noise:", noise.numpy().flatten())
print("Noisy targets:", y_noisy.numpy().flatten())

Original targets: [3. 5. 7. 9.]
Random noise: [ 0.00790854 -0.79504347  0.05179723  0.40587297]
Noisy targets: [3.0079086 4.2049565 7.0517974 9.405873 ]


### 2.1 Full-Batch and Mini-Batch Training

All experiments use the same initial parameters, learning rate,
and number of epochs to ensure a consistent comparison.

In [6]:
# Use identical starting parameters for all experiments
tf.random.set_seed(0)
initial_W = tf.random.normal((1, 1))
initial_b = tf.zeros((1,))

def train_tensorflow(x_data, y_data, batch_size,
                     learning_rate=0.01, epochs=300):

    W = tf.Variable(initial_W)
    b = tf.Variable(initial_b)

    history = []

    dataset = tf.data.Dataset.from_tensor_slices(
        (x_data, y_data)
    )

    # Shuffle observations before mini-batch training
    if batch_size < len(x_data):
        dataset = dataset.shuffle(
            buffer_size=len(x_data),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.batch(batch_size)

    for epoch in range(epochs):

        for x_batch, y_batch in dataset:

            # Record forward pass and loss
            with tf.GradientTape() as tape:
                predictions = tf.matmul(x_batch, W) + b
                loss = tf.reduce_mean(
                    tf.square(predictions - y_batch)
                )

            # Compute gradients and update parameters
            dW, db = tape.gradient(loss, [W, b])

            W.assign_sub(learning_rate * dW)
            b.assign_sub(learning_rate * db)

        # Evaluate loss on the entire dataset after each epoch
        epoch_loss = tf.reduce_mean(
            tf.square(tf.matmul(x_data, W) + b - y_data)
        )

        history.append(float(epoch_loss.numpy()))

    return W.numpy().item(), b.numpy().item(), history

In [7]:
# Experiment 1: Clean full-batch
W_clean, b_clean, clean_history = train_tensorflow(
    x, y, batch_size=4
)

# Experiment 2: Noisy full-batch
W_noisy, b_noisy, noisy_history = train_tensorflow(
    x, y_noisy, batch_size=4
)

# Experiment 3: Noisy mini-batch
W_mini, b_mini, mini_history = train_tensorflow(
    x, y_noisy, batch_size=2
)

results = [
    ("Clean Full-Batch", W_clean, b_clean, clean_history),
    ("Noisy Full-Batch", W_noisy, b_noisy, noisy_history),
    ("Noisy Mini-Batch", W_mini, b_mini, mini_history)
]

for name, weight, bias, history in results:
    print(name)
    print(f"Weight: {weight:.6f}")
    print(f"Bias: {bias:.6f}")
    print(f"Final Loss: {history[-1]:.6f}")
    print("-" * 40)

Clean Full-Batch
Weight: 2.103383
Bias: 0.696042
Final Loss: 0.015430
----------------------------------------
Noisy Full-Batch
Weight: 2.225370
Bias: 0.344835
Final Loss: 0.141706
----------------------------------------
Noisy Mini-Batch
Weight: 2.215482
Bias: 0.376029
Final Loss: 0.141223
----------------------------------------


### 2.2 Comparison of Convergence Curves

In [8]:
fig = go.Figure()

for name, weight, bias, history in results:
    fig.add_trace(go.Scatter(
        x=list(range(1, len(history) + 1)),
        y=history,
        mode="lines",
        name=name
    ))

fig.update_layout(
    title="Gradient Descent: Clean vs Noisy vs Mini-Batch",
    xaxis_title="Epoch",
    yaxis_title="Mean Squared Error (Log Scale)",
    yaxis_type="log",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [9]:
for name, weight, bias, history in results:
    increases = np.sum(np.diff(history) > 1e-8)
    print(f"{name}: {increases} epochs with increasing loss")

Clean Full-Batch: 0 epochs with increasing loss
Noisy Full-Batch: 0 epochs with increasing loss
Noisy Mini-Batch: 114 epochs with increasing loss


### 2.3 Extension Questions

#### Does noise make convergence less smooth?

Random noise introduces variation into the target values,
making the original relationship harder to learn.

The clean full-batch model achieved a final MSE of 0.015430,
while the noisy full-batch model achieved 0.141706.

However, fixed noise in the dataset does not necessarily
make full-batch convergence less smooth. It changes the
loss surface and can result in a higher minimum loss.

#### How does mini-batch training change the behavior of
the optimization path?

Mini-batch training calculates gradients using subsets
of the training data rather than the entire dataset.

This introduces variation in the gradient direction,
which can produce fluctuations in the optimization path.

The noisy mini-batch model achieved a final MSE of 0.141223,
compared with 0.141706 for noisy full-batch training.

Mini-batch training performed 600 parameter updates,
while full-batch training performed 300 updates.
Therefore, the slightly lower final loss does not
necessarily mean mini-batch training is more efficient.

The mini-batch experiment recorded 114 epoch-to-epoch
loss increases, while both full-batch experiments
recorded zero. This demonstrates that mini-batch training
produced a less smooth loss curve.

## 3. Effect of an Excessively Large Learning Rate

This experiment demonstrates how an excessively large learning
rate can cause gradient descent to overshoot the minimum and
diverge instead of converging.

Two learning rates are compared using the same clean dataset
and initial parameters:

- Stable learning rate: 0.01
- Excessively large learning rate: 0.2

In [10]:
test_epochs = 30

# Stable learning rate
W_stable, b_stable, stable_history = train_tensorflow(
    x, y,
    batch_size=4,
    learning_rate=0.01,
    epochs=test_epochs
)

# Excessively large learning rate
W_large, b_large, large_history = train_tensorflow(
    x, y,
    batch_size=4,
    learning_rate=0.2,
    epochs=test_epochs
)

print("Stable Learning Rate (0.01)")
print(f"Initial recorded loss: {stable_history[0]:.6f}")
print(f"Final loss: {stable_history[-1]:.6f}")

print("\nLarge Learning Rate (0.2)")
print(f"Initial recorded loss: {large_history[0]:.6e}")
print(f"Final loss: {large_history[-1]:.6e}")
print(f"Final weight: {W_large:.6e}")
print(f"Final bias: {b_large:.6e}")

Stable Learning Rate (0.01)
Initial recorded loss: 3.662275
Final loss: 0.077993

Large Learning Rate (0.2)
Initial recorded loss: 2.825391e+01
Final loss: 7.339095e+22
Final weight: -8.875640e+10
Final bias: -3.018800e+10


In [11]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(1, test_epochs + 1)),
    y=stable_history,
    mode="lines",
    name="Learning Rate = 0.01"
))

fig.add_trace(go.Scatter(
    x=list(range(1, test_epochs + 1)),
    y=large_history,
    mode="lines",
    name="Learning Rate = 0.2"
))

fig.update_layout(
    title="Effect of Learning Rate on Gradient Descent",
    xaxis_title="Epoch",
    yaxis_title="Mean Squared Error (Log Scale)",
    yaxis_type="log",
    template="plotly_white"
)

fig.show()

## 4. Reflection Questions

### Q7. What happens when the learning rate is too large?

When the learning rate is too large, gradient descent takes
excessively large steps that can overshoot the minimum of
the loss function. This can cause oscillation or divergence
instead of convergence.

In the TensorFlow experiment, we compared a stable learning
rate of 0.01 with a much larger learning rate of 0.2.
An excessively large learning rate can cause the loss to
increase rapidly instead of decreasing.

After 30 epochs, the learning rate of 0.01 produced
a final loss of 0.077993, while the learning rate of
0.2 produced a loss of approximately 7.34 × 10²².

This confirms that an excessively large learning rate
can cause gradient descent to diverge.

### Q8. Why is mini-batch training so common in deep learning?

Mini-batch training provides a balance between computational
efficiency and gradient accuracy.

Instead of processing the entire dataset before updating
the parameters, the model processes smaller groups of samples.
This reduces memory requirements and allows more frequent
parameter updates.

In our experiment, mini-batch training performed two updates
per epoch, compared with one update for full-batch training.
The gradients calculated from different batches can also
introduce fluctuations in the optimization path.

### Q11. In TensorFlow, what is the role of tf.GradientTape()?

GradientTape records differentiable operations performed
during the forward pass and loss calculation.

It uses automatic differentiation to calculate the gradients
of the loss with respect to trainable parameters.

In our implementation, tape.gradient(loss, [W, b]) returns
the gradients for the weight and bias. These gradients are
then used to manually update the parameters using assign_sub().

GradientTape calculates the gradients but does not update
the model parameters itself.

## 5. Experimental Insights

### Insight 1: Low loss does not guarantee exact parameters

After 300 epochs, the clean full-batch model achieved an MSE
of 0.015430. However, the learned weight was 2.103383 and
the bias was 0.696042, compared with the true values of
2.0 and 1.0.

This demonstrates that a model can produce accurate predictions
even before its parameters fully converge.

### Insight 2: Noise affects model accuracy

Adding random noise increased the final full-batch training
loss from 0.015430 to 0.141706.

The learned parameters also moved further away from the
original relationship y = 2x + 1. This demonstrates how
noise in training data can affect parameter estimation.

### Insight 3: Mini-batch training changes update frequency

The noisy mini-batch model achieved a final loss of 0.141223,
slightly lower than the noisy full-batch loss of 0.141706.

However, mini-batch training performed twice as many parameter
updates over the same number of epochs. A fair comparison
should therefore consider both the number of epochs and
the number of parameter updates.